# Challenge AI Engineer

## Daftar Isi

- **Soal 1B**
- **Soal 2**
- **Soal 3**

## Highlights

- Implemented a CLI-style FAQ chatbot using Ollama Cloud models, added model selection so the user can choose which cloud model to use.

## Soal 1B

In [9]:
!pip -q install ollama requests
!pip -q install python-dotenv


In [12]:
import os
import re
from difflib import SequenceMatcher
from dotenv import load_dotenv
from pathlib import Path

import requests
from ollama import Client


### Disini nanti penjelasan ollama api key

In [15]:
OLLAMA_API_KEY = "f6a763daaa0546cea30cfdb4ff9bb474.LKi0Qyrj14T0U7mvye2Ad9Ir"
OLLAMA_HEADERS = {"Authorization": f"Bearer {OLLAMA_API_KEY}"}

### 2) Pilih Model Ollama Cloud

Notebook ini mencoba mengambil daftar model cloud dari `https://ollama.com/api/tags`. Jika daftar tidak bisa diambil, notebook tetap menyediakan pilihan contoh model cloud.

Contoh model cloud yang umum dipakai:

- `gpt-oss:120b-cloud`
- `glm-5:cloud`
- `kimi-k2.5:cloud`


In [16]:
DEFAULT_CLOUD_MODELS = [
    'gpt-oss:120b-cloud',
    'glm-5:cloud',
    'kimi-k2.5:cloud',
]

def fetch_cloud_models():
    try:
        response = requests.get(
            'https://ollama.com/api/tags',
            headers=OLLAMA_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        models = [item.get('name') for item in payload.get('models', []) if item.get('name')]
        return models or DEFAULT_CLOUD_MODELS
    except Exception as exc:
        print(f'Gagal mengambil daftar model cloud: {exc}')
        return DEFAULT_CLOUD_MODELS

available_models = fetch_cloud_models()
print('\nDaftar model yang tersedia:')
for idx, model_name in enumerate(available_models, start=1):
    print(f'{idx}. {model_name}')

choice = input('\nPilih model dengan nomor, atau ketik nama model langsung: ').strip()
if choice.isdigit() and 1 <= int(choice) <= len(available_models):
    selected_model = available_models[int(choice) - 1]
else:
    selected_model = choice or available_models[0]

print(f'Model aktif: {selected_model}')



Daftar model yang tersedia:
1. minimax-m2.7
2. minimax-m3
3. ministral-3:3b
4. gemma3:4b
5. gemma3:27b
6. qwen3-next:80b
7. gpt-oss:120b
8. minimax-m2
9. gemini-3-flash-preview
10. gemma3:12b
11. rnj-1:8b
12. gpt-oss:20b
13. minimax-m2.5
14. ministral-3:14b
15. qwen3.5:397b
16. nemotron-3-ultra
17. glm-5
18. kimi-k2:1t
19. qwen3-coder:480b
20. mistral-large-3:675b
21. devstral-small-2:24b
22. gemma4:31b
23. kimi-k2.5
24. kimi-k2-thinking
25. deepseek-v4-pro
26. deepseek-v3.2
27. glm-5.1
28. deepseek-v4-flash
29. minimax-m2.1
30. nemotron-3-super
31. glm-4.6
32. kimi-k2.6
33. qwen3-coder-next
34. deepseek-v3.1:671b
35. nemotron-3-nano:30b
36. qwen3-vl:235b-instruct
37. ministral-3:8b
38. devstral-2:123b
39. cogito-2.1:671b
40. glm-4.7
41. qwen3-vl:235b

Pilih model dengan nomor, atau ketik nama model langsung: 1
Model aktif: minimax-m2.7


### 3) Buat File `faq.txt`

Setiap baris berisi satu pasangan pertanyaan dan jawaban. Format yang dipakai adalah:

`pertanyaan | jawaban`

Struktur ini mudah dibaca dan mudah diproses untuk mencari jawaban yang relevan.


In [18]:
faq_entries = [
    ('Apa itu notebook ini?', 'Notebook ini adalah submission untuk tes AI Engineer Intern yang berisi chatbot FAQ berbasis Ollama Cloud.'),
    ('Bagaimana cara menjalankan chatbot?', 'Jalankan cell dari atas ke bawah, pilih model, lalu ketik pertanyaan di bagian CLI/Terminal notebook.'),
    ('Apa fungsi file faq.txt?', 'File faq.txt menyimpan konteks FAQ yang menjadi satu-satunya sumber jawaban chatbot.'),
    ('Apa yang dilakukan jika pertanyaan tidak ada di FAQ?', 'Chatbot harus menjawab: Maaf, saya tidak dapat membantu dengan pertanyaan itu.'),
    ('Apakah chatbot ini boleh menjawab dari luar konteks?', 'Tidak. Chatbot hanya boleh menjawab berdasarkan konteks yang ada di faq.txt.'),
    ('Model apa yang bisa dipakai?', 'Anda bisa memilih model cloud yang tersedia, misalnya gpt-oss:120b-cloud, glm-5:cloud, atau kimi-k2.5:cloud jika tersedia di akun Anda.'),
]

faq_path = Path('faq.txt')
faq_path.write_text('\n'.join(f'{q} | {a}' for q, a in faq_entries), encoding='utf-8')
print(f'faq.txt dibuat di: {faq_path.resolve()}')


faq.txt dibuat di: /content/faq.txt


### 4) Load FAQ dan Cari Jawaban yang Relevan

Agar bot tetap ketat, notebook ini melakukan pencarian FAQ paling mirip terlebih dahulu. Jika tingkat kemiripan terlalu rendah, bot langsung memberikan jawaban fallback tanpa menebak.


In [19]:
def load_faq(path='faq.txt'):
    pairs = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or '|' not in line:
            continue
        question, answer = [part.strip() for part in line.split('|', 1)]
        pairs.append((question, answer))
    return pairs


def tokenize(text):
    return set(re.findall(r'[a-z0-9]+', text.lower()))


def similarity(a, b):
    tokens_a = tokenize(a)
    tokens_b = tokenize(b)
    if not tokens_a or not tokens_b:
        return 0.0
    jaccard = len(tokens_a & tokens_b) / len(tokens_a | tokens_b)
    ratio = SequenceMatcher(None, a.lower(), b.lower()).ratio()
    return (0.65 * jaccard) + (0.35 * ratio)


def find_best_faq_match(question, faq_pairs):
    scored = []
    for faq_question, faq_answer in faq_pairs:
        scored.append((similarity(question, faq_question), faq_question, faq_answer))
    scored.sort(reverse=True, key=lambda item: item[0])
    return scored[0] if scored else (0.0, '', '')


faq_pairs = load_faq()
print(f'Jumlah FAQ: {len(faq_pairs)}')
for q, a in faq_pairs:
    print(f'- {q} -> {a}')


Jumlah FAQ: 6
- Apa itu notebook ini? -> Notebook ini adalah submission untuk tes AI Engineer Intern yang berisi chatbot FAQ berbasis Ollama Cloud.
- Bagaimana cara menjalankan chatbot? -> Jalankan cell dari atas ke bawah, pilih model, lalu ketik pertanyaan di bagian CLI/Terminal notebook.
- Apa fungsi file faq.txt? -> File faq.txt menyimpan konteks FAQ yang menjadi satu-satunya sumber jawaban chatbot.
- Apa yang dilakukan jika pertanyaan tidak ada di FAQ? -> Chatbot harus menjawab: Maaf, saya tidak dapat membantu dengan pertanyaan itu.
- Apakah chatbot ini boleh menjawab dari luar konteks? -> Tidak. Chatbot hanya boleh menjawab berdasarkan konteks yang ada di faq.txt.
- Model apa yang bisa dipakai? -> Anda bisa memilih model cloud yang tersedia, misalnya gpt-oss:120b-cloud, glm-5:cloud, atau kimi-k2.5:cloud jika tersedia di akun Anda.


### 5) Jalankan Chatbot CLI/Terminal

Chatbot berikut memakai model Ollama Cloud yang Anda pilih. Respons dijaga tetap terbatas pada konteks FAQ yang ditemukan.

Ketik `exit` atau `quit` untuk mengakhiri percakapan.


In [20]:
client = Client(
    host='https://ollama.com',
    headers=OLLAMA_HEADERS,
)

FALLBACK_ANSWER = 'Maaf, saya tidak dapat membantu dengan pertanyaan itu.'

SYSTEM_PROMPT = (
    'Anda adalah chatbot FAQ yang sangat ketat. '
    'Gunakan hanya konteks FAQ yang diberikan. '
    'Jangan menambah fakta baru, jangan mengarang, dan jangan menjawab di luar konteks. '
    f'Jika konteks tidak cukup, jawab persis: {FALLBACK_ANSWER}'
)


def answer_question(question):
    score, matched_question, matched_answer = find_best_faq_match(question, faq_pairs)
    if score < 0.20:
        return FALLBACK_ANSWER

    context = (
        'Konteks FAQ yang relevan:\n'
        f'Pertanyaan FAQ: {matched_question}\n'
        f'Jawaban FAQ: {matched_answer}\n\n'
        f'Pertanyaan pengguna: {question}\n'
        'Jawab singkat dan hanya berdasarkan jawaban FAQ di atas.'
    )

    try:
        stream = client.chat(
            model=selected_model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': context},
            ],
            stream=True,
        )
        chunks = []
        for part in stream:
            delta = part.get('message', {}).get('content', '')
            if delta:
                chunks.append(delta)
        response_text = ''.join(chunks).strip()
        return response_text or matched_answer
    except Exception as exc:
        print(f'\n[Fallback karena error Ollama Cloud: {exc}]')
        return matched_answer


def run_chatbot():
    print('\nChatbot FAQ siap digunakan.')
    print(f'Model aktif: {selected_model}')
    print('Ketik exit atau quit untuk keluar.')

    while True:
        user_input = input('\nUser: ').strip()
        if user_input.lower() in {'exit', 'quit'}:
            print('Bot: Terima kasih. Sesi chatbot selesai.')
            break
        if not user_input:
            print('Bot: Silakan masukkan pertanyaan.')
            continue

        print('Bot: ', end='', flush=True)
        response = answer_question(user_input)
        print(response)


run_chatbot()



Chatbot FAQ siap digunakan.
Model aktif: minimax-m2.7
Ketik exit atau quit untuk keluar.

User: halo
Bot: Maaf, saya tidak dapat membantu dengan pertanyaan itu.

User: gunanya faq.txt
Bot: 
[Fallback karena error Ollama Cloud: this model requires a subscription, upgrade for access: https://ollama.com/upgrade (ref: 356b4d49-da92-4640-8fae-d148b85fe59a) (status code: 403)]
File faq.txt menyimpan konteks FAQ yang menjadi satu-satunya sumber jawaban chatbot.

User: gunanya faq.txt
Bot: 
[Fallback karena error Ollama Cloud: this model requires a subscription, upgrade for access: https://ollama.com/upgrade (ref: 5cc8405e-82cc-4431-8b01-d3d0a21c1ee7) (status code: 403)]
File faq.txt menyimpan konteks FAQ yang menjadi satu-satunya sumber jawaban chatbot.


KeyboardInterrupt: Interrupted by user

## Bagian 2 - Teori AI

### Soal 2: Pertanyaan Teori Dasar

**a. Apa yang dimaksud dengan Artificial Intelligence (AI)? Sebutkan dua contohnya dalam kehidupan sehari-hari.**

Artificial Intelligence (AI) adalah bidang ilmu komputer yang membuat mesin mampu meniru kemampuan cerdas manusia, seperti mengenali pola, memahami bahasa, mengambil keputusan, dan belajar dari data.

Dua contoh AI dalam kehidupan sehari-hari:

- Rekomendasi video di YouTube atau Netflix.
- Asisten virtual seperti Siri, Google Assistant, atau ChatGPT.

**b. Apa perbedaan antara Supervised Learning dan Unsupervised Learning? Berikan satu contoh untuk masing-masing.**

- **Supervised Learning** menggunakan data yang sudah memiliki label jawaban. Model belajar dari pasangan input-output yang benar. Contoh: klasifikasi email spam dan bukan spam.
- **Unsupervised Learning** menggunakan data tanpa label. Model mencari pola atau struktur sendiri. Contoh: clustering pelanggan berdasarkan perilaku belanja.

### Soal 3: Pertanyaan Konsep

**a. Apa itu Feature dalam konteks machine learning? Mengapa penting untuk memilih fitur yang tepat saat membangun model?**

Feature adalah variabel atau atribut yang digunakan model sebagai masukan untuk mempelajari pola. Contohnya umur, pendapatan, atau jumlah klik.

Pemilihan fitur yang tepat penting karena fitur yang relevan membantu model belajar lebih akurat, lebih cepat, dan lebih stabil. Fitur yang kurang tepat dapat membuat model sulit belajar atau menghasilkan prediksi yang kurang baik.

**b. Apa itu Fine-tuning dalam machine learning? Sebutkan satu kasus di mana fine-tuning berguna.**

Fine-tuning adalah proses menyesuaikan model yang sudah pre-trained agar lebih cocok dengan tugas atau data yang lebih spesifik.

Contoh kasus yang berguna: model bahasa umum di-fine-tune untuk klasifikasi sentimen ulasan pelanggan pada domain e-commerce atau layanan keuangan.

### Kesimpulan

Notebook ini menunjukkan implementasi chatbot FAQ berbasis Ollama Cloud yang hanya menjawab dari konteks `faq.txt`, serta jawaban teori AI dasar dalam format yang siap dijalankan di Google Colab.
